# LC 417 — Pacific Atlantic Water Flow
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Graphs
**Pattern:** Reverse Multi-Source BFS/DFS from Borders

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Instead of asking
"can water flow from this cell to both oceans?",
reverse it: BFS inward from each ocean's border
marking reachable cells. The answer is every cell
reachable from both oceans.
</div>

## Official Problem Statement

There is an `m x n` rectangular island that
borders both the **Pacific** and **Atlantic**
oceans. The Pacific touches the island's top and
left edges; the Atlantic touches the bottom and
right edges.

Water can only flow to neighbouring cells with
height **equal or lower**. Return all cells from
which water can flow to both the Pacific and
Atlantic oceans.

**Example 1:**
```
Input:
  heights = [[1,2,2,3,5],
              [3,2,3,4,4],
              [2,4,5,3,1],
              [6,7,1,4,5],
              [5,1,1,2,4]]
Output: [[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]
```

**Constraints:**
- `m == heights.length`, `n == heights[0].length`
- `1 <= m, n <= 200`
- `0 <= heights[i][j] <= 10^5`

## What This Is Actually Asking

Rain falls on a grid of mountains. Water flows
downhill (or stays level). The left/top edges
drain to the Pacific; the right/bottom edges drain
to the Atlantic. Find every cell where rain water
can eventually reach both oceans.

## Walk Through an Example by Hand

```
heights = [[1,2,2,3,5],
           [3,2,3,4,4],
           [2,4,5,3,1],
           [6,7,1,4,5],
           [5,1,1,2,4]]

Reverse BFS — water flows UPHILL from oceans:

Pacific starts: top row + left col
  seeds = all (0,j) and (i,0)
  BFS: from each seed visit neighbours with
       height >= current (water can flow to us)
  pacific_reachable = set of cells

Atlantic starts: bottom row + right col
  seeds = all (m-1,j) and (i,n-1)
  Same BFS uphill
  atlantic_reachable = set of cells

Answer = pacific_reachable ∩ atlantic_reachable

Cell (2,2) height=5 — highest point:
  Can drain left/up to Pacific? Yes
  Can drain right/down to Atlantic? Yes
  -> in answer
```

## The Picture

```
PACIFIC (P) touches top + left
ATLANTIC (A) touches bottom + right

  P P P P P
P [1,2,2,3,5] A
P [3,2,3,4,4] A
P [2,4,5,3,1] A
P [6,7,1,4,5] A
P [5,1,1,2,4] A
  A A A A A

Naive approach: from each cell run DFS to check
  both oceans -> O(m*n*(m+n)) — too slow

Reverse approach: BFS INWARD from each border
  Pacific BFS: can water flow back from border
  uphill? Neighbour height >= current -> reachable

  Mark pacific set (P) and atlantic set (A).
  Output = cells in both P and A.

  [*,*,*,*,P]      [*,*,*,*,A]
  [*,*,*,P,P]      [A,A,A,A,A]
  [*,*,P,*,*]  +   [A,A,*,*,A]
  [P,P,*,*,*]      [A,A,*,A,A]
  [P,*,*,*,*]      [A,*,*,A,A]
  Intersection = answer cells
```

## When To Use This Pattern

- When a forward search from every cell is too
  slow, think **reverse the search — BFS inward
  from the destination borders**
- When two conditions must both be true, think
  **two separate BFS passes, then intersect**
- When flow goes downhill, think
  **reverse: go uphill — visit neighbour if
  height >= current**
- When seeding multiple border cells at once,
  think **multi-source BFS — add all seeds to
  queue before starting**

## The Approach

Create two visited sets: pacific and atlantic.
Seed the pacific BFS with all top-row and left-
column cells. Seed the atlantic BFS with all
bottom-row and right-column cells. In each BFS,
visit a neighbour only if its height is at or
above the current cell (water can flow toward us).
Return cells present in both visited sets.

In [ ]:
from collections import deque  # BFS queue
from typing import List

In [ ]:
def test_harness(func):
    tests = [
        (
            [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],
             [6,7,1,4,5],[5,1,1,2,4]],
            sorted([[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]])
        ),
        ([[1]], [[0,0]]),           # 1x1 grid
        ([[1,1],[1,1]], sorted([[0,0],[0,1],[1,0],[1,1]])),
        ([[3,3,3],[3,1,3],[0,2,4]],
         sorted([[0,0],[0,1],[0,2],[1,0],[1,2],[2,2]])),
    ]

    passed = 0
    for i, (heights, expected) in enumerate(tests):
        result = sorted(func([r[:] for r in heights]))
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def pacificAtlantic(
    heights: List[List[int]]
) -> List[List[int]]:
    """
    Return cells from which water reaches both oceans.

    Two BFS passes — Pacific (top+left borders) and
    Atlantic (bottom+right borders). Move to neighbours
    with height >= current (reverse flow direction).
    Return intersection of both visited sets.

    Time:  O(m*n) — each cell visited at most twice
    Space: O(m*n) — two visited sets
    """
    pass


# Quick debug — run this cell while building
h = [[1,2,2,3,5],[3,2,3,4,4],[2,4,5,3,1],
     [6,7,1,4,5],[5,1,1,2,4]]
print(sorted(pacificAtlantic(h)))
# [[0,4],[1,3],[1,4],[2,2],[3,0],[3,1],[4,0]]
print(pacificAtlantic([[1]]))   # [[0,0]]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(pacificAtlantic)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Forward DFS from every cell | O(m*n*(m+n)) | O(m*n) |
| Reverse BFS from borders | O(m*n) | O(m*n) |

Reversing the direction turns an expensive multi-
source forward search into two simple BFS passes —
each cell is processed at most once per pass.

## Real World Connection

At Citi, data lineage analysis asks: which source
tables feed both the risk report AND the compliance
report? Forward tracing from every source is
expensive on a graph with thousands of nodes.
The reverse approach — BFS backwards from each
report's output node — finds its full upstream set
in O(V+E). The intersection is the shared lineage.
On AWS Glue, the same reverse-BFS on the data
catalog lineage graph identifies which S3 datasets
feed multiple downstream pipelines simultaneously.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra